In [2]:
import numpy as np
import matplotlib.pyplot as plt
import cmocean
from netCDF4 import Dataset
from mpl_toolkits.basemap import Basemap
from matplotlib import colors

# -- Load Sen's slope --
file_id = Dataset('../../data/chl/chl_senslope_summer_20260328.nc')
slope = np.array(file_id.variables["slope"][:], dtype='float64')
lat   = file_id.variables["latitude"][:].copy()
lon   = file_id.variables["longitude"][:].copy()
file_id.close()

# -- Load Kendall tau p-values --
file_id = Dataset('../../data/chl/chl_kendall_summer_20260328.nc')
pval = np.array(file_id.variables["pvalue"][:], dtype='float64')
file_id.close()

# Convert slope to per year
slope_yr = slope * 365.25

# White out non-significant pixels
slope_sig = np.where(pval <= 0.05, slope_yr, np.nan)

# -- Diverging norm centered on zero --
divnorm = colors.TwoSlopeNorm(vmin=np.nanmin(slope_sig),
                               vcenter=0,
                               vmax=np.nanmax(slope_sig))

# -- Set up grids --
lon_grid, lat_grid = np.meshgrid(lon, lat)

FileNotFoundError: [Errno 2] No such file or directory: '../../data/chl/chl_senslope_summer_20260328.nc'

In [ ]:
# -- Plot --
fig, ax = plt.subplots(1, 1, figsize=(15, 7))

m = Basemap(projection='lcc', resolution='l',
            llcrnrlat=16, urcrnrlat=35.5,
            llcrnrlon=-170, urcrnrlon=-130,
            lat_0=30, lon_0=-150,
            width=5.1E6, height=5E6,
            ax=ax)

x, y = m(lon_grid, lat_grid)

contour = m.contourf(x, y, slope_sig, cmap=cmocean.cm.tarn_r,
                     levels=100, extend='both', norm=divnorm)

m.fillcontinents(color='black')
m.drawcoastlines(color='black')

m.drawparallels(np.arange(18, 40, 2), labels=[1, 0, 0, 0],
                textcolor='black', color='dimgrey', fontsize=10,
                dashes=(3, 5), linewidth=0.6)
m.drawmeridians(np.arange(-175, -130, 3), labels=[0, 0, 0, 1],
                textcolor='black', color='dimgrey', fontsize=10,
                dashes=(3, 5), linewidth=0.6)

cbar = fig.colorbar(contour, shrink=0.965, pad=0.02)
cbar.set_label('mg m$^{-3}$ year$^{-1}$', fontsize=11)

plt.title("Sen's slope of summertime chlorophyll anomaly (p < 0.05) from 1998 to 2025 [mg m$^{-3}$ year$^{-1}$]",
          fontsize=13)

plt.savefig('senslope_significant_map.png', bbox_inches='tight', dpi=300)
plt.show()